In [ ]:
from __future__ import annotations
import os
import tempfile
from itertools import chain
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from notebooks.utils.bounded_prefix_evaluation_utils import (
    PREFIX_COVERAGE_COLUMNS, PREFIX_METRIC_COLUMNS, PREFIX_RUN_SUMMARY_COLUMNS,
    bounded_prefix_prediction_key, bounded_prefix_successful_keys,
    collect_source_parent_identities, ensure_bounded_prefix_result_exports,
    iter_bounded_prefix_feature_batches, prefix_window_label, validate_prefix_bounds,
)
from notebooks.utils.classification_experiment_utils import (
    DATASETS, EVALUATION_REGIMES as EVALUATION_REGIME_SOURCES, append_result_frame, build_feature_parent_cohorts,
    discover_model_artifacts,
)
from notebooks.utils.feature_aggregation_utils import FeatureSet
from notebooks.utils.model_training_utils import (
    load_model_artifact, load_model_metadata, score_feature_vector_model,
)
from notebooks.utils.streaming_classification_evaluation_utils import (
    aggregate_prediction_artifact, prediction_artifact_is_complete, prediction_artifact_path,
    write_prediction_artifact,
)
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
MODELS_ROOT = DATA_ROOT / "experiment-results" / "classification" / "models"
RUN_ID = "bounded-prefix-classifier-evaluation-gemma270m-rf-window32-2026-09-29"
# RUN_ID = "bounded-prefix-classifier-evaluation-smollm2_360m-rf-window32-2026-09-29"

SOURCE_RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
MODEL_FAMILIES = ("random_forest",)
SCORER_LLMS = ("gemma_3_270m_it",)
# SCORER_LLMS = ("smollm2_360m",)

FEATURE_SETS = ("simple_similarity", "scalar_all")
TRAINING_WINDOW = "32"
TRAINING_REGIMES = (
    "Kaggle+MAGE+Outfox",
    "Kaggle+MAGE+Outfox+BBC",
)
EVALUATION_REGIMES = ("Mixed",)
SOURCE_SPLITS = {"Kaggle": ("HG", "MG"), "MAGE": ("HG", "MG"), "Outfox": ("HG", "MG"), "BBC": ("HG",)}
TOKEN_BOUNDS = ((100, 300), (300, 600), (600, 1200))
PARTITIONS = ("validation",)  # add "test" to score held-out test cohorts as well
EVALUATION_BATCH_SIZE = 256
MIN_MACRO_F1 = 0.80
MIN_BALANCED_ACCURACY = 0.80
MIN_CLASS_SUPPORT = 25
for minimum, maximum in TOKEN_BOUNDS:
    validate_prefix_bounds(minimum, maximum)
exports = ensure_bounded_prefix_result_exports(DATA_ROOT, RUN_ID)
result_key_columns = ("artifact_id", "evaluation_regime", "dataset", "partition", "min_tokens", "max_tokens")
completed_key_sets = [
    bounded_prefix_successful_keys(exports[name], result_key_columns)
    for name in ("prefix_evaluation_metrics", "prefix_evaluation_coverage")
]
completed_result_keys = set.intersection(*completed_key_sets)
artifact_paths = discover_model_artifacts(MODELS_ROOT, filters={"run_id": SOURCE_RUN_ID})


In [ ]:
for artifact_path in tqdm(artifact_paths, desc="Evaluating bounded prefixes", unit="artifact"):
    try:
        metadata = load_model_metadata(artifact_path, expected={"run_id": SOURCE_RUN_ID})
        model_family = str(metadata["model_family"])
        scorer_llm = str(metadata["scorer_llm"])
        feature_set_name = str(metadata["feature_set"])
        feature_columns = tuple(metadata["feature_columns"])
        if (
            model_family not in MODEL_FAMILIES
            or scorer_llm not in SCORER_LLMS
            or feature_set_name not in FEATURE_SETS
            or str(metadata.get("training_window")) != TRAINING_WINDOW
        ):
            continue
        fitted = load_model_artifact(
            artifact_path,
            expected={"model_family": model_family, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "feature_columns": list(feature_columns)},
            trust_artifact=True,
        )
    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
        append_result_frame(exports["prefix_run_summary"], pd.DataFrame([{
            "run_id": RUN_ID, "stage": "artifact", "status": "unavailable", "artifact_id": "",
            "artifact_path": str(artifact_path), "model_family": "", "scorer_llm": "", "feature_set": "",
            "training_regime": "", "training_window": "", "evaluation_regime": "", "dataset": "",
            "partition": "", "min_tokens": pd.NA, "max_tokens": pd.NA, "error": str(error),
        }], columns=PREFIX_RUN_SUMMARY_COLUMNS))
        continue
    training_regime = str(metadata["training_regime"])
    if training_regime not in TRAINING_REGIMES:
        continue
    training_window = str(metadata["training_window"])
    feature_set = FeatureSet(feature_set_name, feature_columns)
    artifact_dimensions = {
        "run_id": RUN_ID, "artifact_path": str(artifact_path), "model_family": model_family,
        "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": training_regime,
        "training_window": training_window,
    }
    for evaluation_regime in tqdm(EVALUATION_REGIMES, desc=f"{model_family} regimes", unit="regime", leave=False):
        source_names = EVALUATION_REGIME_SOURCES[evaluation_regime]
        parent_rows = []
        try:
            for source_name in source_names:
                dataset = DATASETS[source_name]
                for split in SOURCE_SPLITS[source_name]:
                    parent_rows.append(collect_source_parent_identities(
                        DATA_ROOT / "processed", scorer_llm=scorer_llm, dataset=dataset, split=split,
                        max_per_split=int(metadata.get("max_per_split", 600)), min_text_tokens=int(metadata.get("min_text_tokens", 100)), batch_size=EVALUATION_BATCH_SIZE,
                        progress_description=f"Collecting parents: {scorer_llm}/{dataset}/{split}",
                    ))
            parent_frame = pd.concat(parent_rows, ignore_index=True)
            cohorts = build_feature_parent_cohorts(
                parent_frame, seed=int(metadata.get("cohort_seed", 42)),
                max_per_split=int(metadata.get("max_per_split", 600)), min_text_tokens=int(metadata.get("min_text_tokens", 100)),
            )
        except (FileNotFoundError, KeyError, ValueError, OSError) as error:
            append_result_frame(exports["prefix_run_summary"], pd.DataFrame([{
                **artifact_dimensions, "stage": "cohort", "status": "unavailable", "artifact_id": "",
                "evaluation_regime": evaluation_regime, "dataset": "|".join(source_names), "partition": "",
                "min_tokens": pd.NA, "max_tokens": pd.NA, "error": str(error),
            }], columns=PREFIX_RUN_SUMMARY_COLUMNS))
            continue
        for partition in PARTITIONS:
            selected_cohort = cohorts[partition]
            for minimum, maximum in TOKEN_BOUNDS:
                for source_name in source_names:
                    dataset = DATASETS[source_name]
                    artifact_key = bounded_prefix_prediction_key(
                        artifact_path, models_root=MODELS_ROOT, evaluation_regime=evaluation_regime, dataset=dataset,
                        partition=partition, min_tokens=minimum, max_tokens=maximum,
                    )
                    result_key = tuple(str(artifact_key[column]) for column in result_key_columns)
                    prediction_path = prediction_artifact_path(exports["prefix_evaluation_metrics"].parent, artifact_key)
                    if result_key in completed_result_keys and prediction_artifact_is_complete(prediction_path, expected=artifact_key):
                        continue
                    selected_ids_by_split = {
                        split: {identity[2] for identity in selected_cohort if identity[:2] == (dataset, split)}
                        for split in SOURCE_SPLITS[source_name]
                    }
                    cohort_parent_count = sum(len(ids) for ids in selected_ids_by_split.values())
                    dimensions = {**artifact_dimensions, **artifact_key}
                    status = "unavailable"
                    error = ""
                    try:
                        if not prediction_artifact_is_complete(prediction_path, expected=artifact_key):
                            scored_batches = (
                                score_feature_vector_model(fitted, batch).assign(**dimensions)
                                for split, selected_ids in selected_ids_by_split.items() if selected_ids
                                for batch in iter_bounded_prefix_feature_batches(
                                    DATA_ROOT / "processed", scorer_llm=scorer_llm, dataset=dataset, split=split,
                                    feature_set=feature_set, min_tokens=minimum, max_tokens=maximum,
                                    selected_ids=selected_ids, batch_size=EVALUATION_BATCH_SIZE,
                                    progress_description=f"Scoring prefix: {evaluation_regime}/{partition}/{minimum}-{maximum}/{dataset}/{split}",
                                )
                            )
                            first_batch = next(scored_batches, None)
                            if first_batch is None:
                                raise ValueError("no eligible prefix rows for this bound and cohort")
                            if not {"original_token_count", "evaluated_token_count"}.issubset(first_batch.columns):
                                raise ValueError("bounded-prefix rows must retain original_token_count and evaluated_token_count")
                            write_prediction_artifact(
                                prediction_path, expected=artifact_key, batches=chain((first_batch,), scored_batches),
                            )
                        prediction_rows = pq.ParquetFile(prediction_path / "data.parquet").read(columns=["split", "text_id"]).to_pandas()
                        scored_row_count = len(prediction_rows)
                        eligible_parent_count = len(prediction_rows.drop_duplicates(["split", "text_id"]))
                        artifact_metrics = aggregate_prediction_artifact(prediction_path, threshold=0.5)
                        has_both_labels = bool(artifact_metrics["hg_support"] and artifact_metrics["mg_support"])
                        has_required_support = bool(
                            artifact_metrics["hg_support"] >= MIN_CLASS_SUPPORT
                            and artifact_metrics["mg_support"] >= MIN_CLASS_SUPPORT
                        )
                        available = has_both_labels and has_required_support
                        status = "available" if available else "unavailable"
                        error = (
                            "" if available else "eligible prefixes contain one class"
                            if not has_both_labels else "eligible prefixes do not meet minimum class support"
                        )
                        metric_row = {
                            **dimensions, "eligible_parent_count": eligible_parent_count, "scored_row_count": scored_row_count,
                            **{column: artifact_metrics.get(column, pd.NA) if has_both_labels else pd.NA for column in ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "false_positive_rate", "tn", "fp", "fn", "tp")},
                            "hg_support": artifact_metrics["hg_support"], "mg_support": artifact_metrics["mg_support"],
                            "status": status, "error": error,
                        }
                        coverage_row = {
                            **dimensions, "split": "+".join(SOURCE_SPLITS[source_name]),
                            "class": "binary" if available else SOURCE_SPLITS[source_name][0],
                            "cohort_parent_count": cohort_parent_count, "eligible_parent_count": eligible_parent_count,
                            "scored_row_count": scored_row_count, "hg_support": artifact_metrics["hg_support"],
                            "mg_support": artifact_metrics["mg_support"], "availability_status": status, "error": error,
                        }
                        append_result_frame(exports["prefix_evaluation_metrics"], pd.DataFrame([metric_row], columns=PREFIX_METRIC_COLUMNS))
                        append_result_frame(exports["prefix_evaluation_coverage"], pd.DataFrame([coverage_row], columns=PREFIX_COVERAGE_COLUMNS))
                        if available:
                            completed_result_keys.add(result_key)
                    except (FileNotFoundError, KeyError, ValueError, OSError) as failure:
                        error = str(failure)
                        metric_row = {
                            **dimensions, "eligible_parent_count": 0, "scored_row_count": 0,
                            "accuracy": pd.NA, "balanced_accuracy": pd.NA, "macro_f1": pd.NA, "roc_auc": pd.NA,
                            "false_positive_rate": pd.NA, "hg_support": 0, "mg_support": 0,
                            "tn": pd.NA, "fp": pd.NA, "fn": pd.NA, "tp": pd.NA, "status": "unavailable", "error": str(error),
                        }
                        coverage_row = {
                            **dimensions, "split": "+".join(SOURCE_SPLITS[source_name]),
                            "class": "binary", "cohort_parent_count": cohort_parent_count,
                            "eligible_parent_count": 0, "scored_row_count": 0, "hg_support": 0, "mg_support": 0,
                            "availability_status": "unavailable", "error": str(error),
                        }
                        append_result_frame(exports["prefix_evaluation_metrics"], pd.DataFrame([metric_row], columns=PREFIX_METRIC_COLUMNS))
                        append_result_frame(exports["prefix_evaluation_coverage"], pd.DataFrame([coverage_row], columns=PREFIX_COVERAGE_COLUMNS))
                    append_result_frame(exports["prefix_run_summary"], pd.DataFrame([{
                        **dimensions, "stage": "evaluation", "status": status, "error": error,
                    }], columns=PREFIX_RUN_SUMMARY_COLUMNS))


In [ ]:
metrics = pd.read_csv(exports["prefix_evaluation_metrics"])
coverage = pd.read_csv(exports["prefix_evaluation_coverage"])
per_dataset = metrics.sort_values(["model_family", "scorer_llm", "training_regime", "evaluation_regime", "partition", "max_tokens", "min_tokens", "dataset"])
display(per_dataset)
pool_dimensions = ["artifact_id", "model_family", "scorer_llm", "feature_set", "training_regime", "training_window", "evaluation_regime", "partition", "min_tokens", "max_tokens"]
scored_metrics = metrics.loc[metrics["scored_row_count"].gt(0)].copy()
pooled_rows = []
with tempfile.TemporaryDirectory(prefix="bounded-prefix-pooled-") as temporary_directory:
    for values, group in scored_metrics.groupby(pool_dimensions, dropna=False):
        dimensions = dict(zip(pool_dimensions, values, strict=True))
        required_datasets = {DATASETS[source_name] for source_name in EVALUATION_REGIME_SOURCES[dimensions["evaluation_regime"]]}
        component_rows = group.drop_duplicates("dataset", keep="last")
        paths, missing_datasets = [], required_datasets.difference(component_rows["dataset"])
        for row in component_rows.itertuples(index=False):
            artifact_key = {
                "artifact_id": row.artifact_id, "evaluation_regime": row.evaluation_regime,
                "evaluation_window": prefix_window_label(int(row.min_tokens), int(row.max_tokens)),
                "dataset": row.dataset, "partition": row.partition,
                "min_tokens": int(row.min_tokens), "max_tokens": int(row.max_tokens),
            }
            prediction_path = prediction_artifact_path(exports["prefix_evaluation_metrics"].parent, artifact_key)
            if prediction_artifact_is_complete(prediction_path, expected=artifact_key):
                paths.append(prediction_path)
            else:
                missing_datasets.add(row.dataset)
        if missing_datasets or set(component_rows["dataset"]) != required_datasets:
            pooled_rows.append(dimensions | {
                "hg_support": pd.NA, "mg_support": pd.NA, "macro_f1": pd.NA, "balanced_accuracy": pd.NA,
                "status": "unavailable", "error": f"missing complete dataset artifacts: {sorted(missing_datasets)}",
            })
            continue
        pooled_path = Path(temporary_directory) / str(len(pooled_rows))
        write_prediction_artifact(
            pooled_path, expected={},
            batches=(batch.to_pandas() for path in paths for batch in pq.ParquetFile(path / "data.parquet").iter_batches(batch_size=EVALUATION_BATCH_SIZE)),
        )
        pooled_metrics = aggregate_prediction_artifact(pooled_path, threshold=0.5)
        has_required_support = bool(pooled_metrics["hg_support"] >= MIN_CLASS_SUPPORT and pooled_metrics["mg_support"] >= MIN_CLASS_SUPPORT)
        pooled_rows.append(dimensions | pooled_metrics | {
            "status": "available" if has_required_support else "unavailable",
            "error": "" if has_required_support else "pooled prefixes do not meet minimum class support",
        })
pooled = pd.DataFrame(pooled_rows)
comparison = pooled.sort_values(["max_tokens", "min_tokens"]) if not pooled.empty else pooled
display(comparison)
qualified = pooled.loc[pooled["status"].eq("available") & pooled["macro_f1"].ge(MIN_MACRO_F1) & pooled["balanced_accuracy"].ge(MIN_BALANCED_ACCURACY)].sort_values(["max_tokens", "min_tokens"]) if not pooled.empty else pooled
qualification_dimensions = ["artifact_id", "evaluation_regime", "partition"]
qualification_rows = []
for values, group in pooled.groupby(qualification_dimensions, dropna=False) if not pooled.empty else ():
    matches = qualified.loc[(qualified[qualification_dimensions] == pd.Series(values, index=qualification_dimensions)).all(axis=1)]
    qualification_rows.append(dict(zip(qualification_dimensions, values, strict=True)) | (
        {"qualification": "No qualifying length"} if matches.empty else {"qualification": "qualified", "min_tokens": int(matches.iloc[0]["min_tokens"]), "max_tokens": int(matches.iloc[0]["max_tokens"])}
    ))
display(pd.DataFrame(qualification_rows))
display(coverage.sort_values(["max_tokens", "min_tokens", "dataset"]))
